# Решения: практика поиска

**Для преподавателя.** Эталон к `lesson.ipynb` и `homework.ipynb`. Не показывать ученикам до сдачи.

In [ ]:
from pathlib import Path
import time
import pandas as pd


def _find(name: str) -> Path:
    for p in (Path(name), Path(f'../../data/{name}'), Path(f'../data/{name}')):
        if p.exists():
            return p.resolve()
    raise FileNotFoundError(f'{name} не найден рядом с ноутбуком')


unsorted_df = pd.read_csv(_find('bank_transactions_unsorted.csv'))
by_id_df = pd.read_csv(_find('bank_transactions_sorted_by_txn_id.csv'))
by_amount_df = pd.read_csv(_find('bank_transactions_sorted_by_amount.csv'))
tiny_df = pd.read_csv(_find('bank_transactions_tiny.csv'))

unsorted_txns = list(unsorted_df[['txn_id', 'amount', 'day', 'risk_score']].itertuples(index=False, name=None))
id_txns = list(by_id_df[['txn_id', 'amount', 'day', 'risk_score']].itertuples(index=False, name=None))
amount_txns = list(by_amount_df[['txn_id', 'amount', 'day', 'risk_score']].itertuples(index=False, name=None))
id_list = [t[0] for t in id_txns]
amount_list = [t[1] for t in amount_txns]


In [ ]:
def linear_search_txn(txns, target_id):
    for i, row in enumerate(txns):
        if row[0] == target_id:
            return i
    return -1


def binary_search_txn(sorted_ids, target_id):
    left, right = 0, len(sorted_ids) - 1
    while left <= right:
        mid = (left + right) // 2
        if sorted_ids[mid] == target_id:
            return mid
        if sorted_ids[mid] < target_id:
            left = mid + 1
        else:
            right = mid - 1
    return -1


def lower_bound(nums, target):
    left, right = 0, len(nums)
    while left < right:
        mid = (left + right) // 2
        if nums[mid] < target:
            left = mid + 1
        else:
            right = mid
    return left


def upper_bound(nums, target):
    left, right = 0, len(nums)
    while left < right:
        mid = (left + right) // 2
        if nums[mid] <= target:
            left = mid + 1
        else:
            right = mid
    return left


target_id = unsorted_txns[40][0]
idx = linear_search_txn(unsorted_txns, target_id)
row = unsorted_txns[idx]
idx2 = binary_search_txn(id_list, id_list[300])
left_ok = id_list[idx2 - 1] <= id_list[idx2]
right_ok = id_list[idx2] <= id_list[idx2 + 1]
low = lower_bound(amount_list, 5000)
high = upper_bound(amount_list, 12000)
subset = amount_txns[low:high]
RANGE_NOTE = (
    'Если суммы отсортированы, два бинарных поиска быстро находят границы диапазона, '
    'и мы работаем только с нужным фрагментом лога.'
)
count_small = upper_bound(amount_list, 10000)
s_low = lower_bound(amount_list, 15000)
s_high = upper_bound(amount_list, 17000)
sample_ids = [row[0] for row in amount_txns[s_low:s_low + 10]]
WHY_RANGE = (
    'Полный проход проверяет все строки, даже если интересует узкий диапазон. '
    'На отсортированных данных диапазонный поиск сокращает число сравнений и ускоряет отчёт.'
)
print(row)
print(idx2, left_ok, right_ok)
print(low, high, len(subset))